In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from pathlib import Path

import pandas as pd
import numpy as np

In [ ]:
model_id = "sentence-transformers/all-MiniLM-L6-v2"

#revision = HfApi().model_info(model_id).sha
#print(f'revision: {revision}')
#revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41

revision = '1110a243fdf4706b3f48f1d95db1a4f5529b4d41'
encoder = SentenceTransformer(model_id, revision = revision)

print(f'max seq length: {encoder.max_seq_length}')
print(f'embeddings length: {encoder.get_embedding_dimension()}')

In [ ]:
repo_id="EXAONE-BI/FinTexTS"

# dataset_info = HfApi().dataset_info(repo_id, files_metadata=True)
# print(f"dataset's revision: {dataset_info.sha}")
# dataset's revision: 6a4ce04ac3f2e8a4ae15a5826e6980b50560eb29

ds = load_dataset(repo_id, revision="6a4ce04ac3f2e8a4ae15a5826e6980b50560eb29")

In [ ]:
print(ds["train"].features)
text_columns = [name for name, features in ds['train'].features.items() if getattr(features, "dtype", None) =='string']
text_columns = [name for name in text_columns if name != "ticker"]
print(text_columns)

In [ ]:
df_check = ds['train'].select_columns(text_columns).to_pandas().apply(lambda s: s.str.strip().replace("", None))
embeddings_by_column = {}

for column in df_check.columns:
 mask = df_check[column].notna().to_numpy()
 texts = df_check.loc[mask, column].tolist()
 
 embeddings = np.zeros((len(df_check), 384),dtype=np.float32,)
 
if texts:
    embeddings[mask] = encoder.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

embeddings_by_column[column] = embeddings

In [ ]:
groups = {
    "macro": [f"macro_category{i}" for i in range(1, 6)],
    "sector": [f"sector_category{i}" for i in range(1, 6)],
    "targetCompany": [f"targetCompany_category{i}" for i in range(1, 4)],
    "relatedCompany": [f"relatedCompany_category{i}" for i in range(1, 4)],
    "filing": [
        "filing_financialStatement",
        "filing_governanceRisks",
        "filing_overviewProduct",
        "filing_recentEventCatalyst",
        "filing_strategyMarketOps",
    ],
}

level_embeddings = {}
level_masks = {}

for level, columns in groups.items():
    embedding_sum = np.zeros((len(df_check), 384), dtype=np.float32)
    counts = np.zeros(len(df_check), dtype=np.int32)

    for column in columns:
        values = embeddings_by_column[column]
        present = df_check[column].notna().to_numpy()
        embedding_sum[present] += values[present]
        counts += present

    averaged = np.zeros_like(embedding_sum)

    np.divide(
        embedding_sum,
        counts[:, None],
        out=averaged,
        where=counts[:, None] > 0,
    )

    level_embeddings[level] = averaged
    level_masks[level] = counts > 0

In [ ]:
num_columns = ["date", "ticker", "open", "high", "low", "close"]
metadata = (
    ds["train"]
    .select_columns(num_columns)
    .to_pandas()
    .reset_index(drop=True)
)

assert len(metadata) == len(df_check)

for values in level_embeddings.values():
    assert values.shape == (len(metadata), 384)

output_dir = Path(
    "/auto/datasets/kondyurin/FinTexTS/minilm_reproduction_v1/"
)
output_dir.mkdir(parents=True, exist_ok=True)

tickers = metadata["ticker"].unique()

for ticker in tickers:
    file_path = output_dir / f"{ticker}.parquet"
    if file_path.exists():
        raise FileExistsError(f"File already exists: {file_path}")

for ticker in tickers:
    mask = metadata["ticker"].to_numpy() == ticker
    positions = np.flatnonzero(mask)

    data_per_ticker = metadata.iloc[positions].reset_index(drop=True)
    parts = [data_per_ticker]

    for level, values in level_embeddings.items():
        pos = values[positions]
        col = [f"{level}_emb{i}" for i in range(384)]

        dr = pd.DataFrame(pos, columns=col)
        parts.append(dr)

    result = pd.concat(parts, axis=1)
    result = result.sort_values("date").reset_index(drop=True)

    assert not result["date"].duplicated().any()
    assert not result.isna().any().any()

    file_path = output_dir / f"{ticker}.parquet"
    result.to_parquet(file_path, index=False)

print("Saved files:", len(tickers))